In [ ]:
import os
import json
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from cycler import cycler
import re
# import matplotlib.pyplot as plt
# from cycler import cycler

colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple',
          'tab:brown', 'tab:pink', 'tab:gray', 'tab:olive', 'tab:cyan',
          'darkblue', 'darkorange', 'darkgreen', 'darkred', 'darkviolet']  # Added 5 more colors

linestyles = ['-', '--', '-.', ':',
              (0, (3, 1, 1, 1)), (0, (5, 5)), (0, (1, 1)), (0, (5, 2, 1, 2)), (0, (2, 2, 3, 1)), (0, (4, 1, 2, 1)),
              (0, (3, 1, 3, 1)), (0, (3, 2)), (0, (1, 2)), (0, (5, 1)), (0, (3, 5))]  # Added 5 more linestyles

# Create a combined cycler with 15 unique combinations
style_cycler = cycler(color=colors[:15], linestyle=linestyles[:15])
plt.rc('axes', prop_cycle=style_cycler)

In [ ]:
def extract_timed_results_dir(base_dir):
    directories = os.listdir(base_dir)
    out = list()
    for directory in directories:
        splitted = directory.split("_")
        if len(splitted) > 1:
            try:
                int(splitted[-1])
                out.append(os.path.join(base_dir, directory))
            except Exception as e:
                continue
    return out


In [ ]:
BASE_DIR = "results_27122025"
APPROACHES = [name for name in os.listdir(BASE_DIR)]
EXCLUDED = ['ttt',
            'tic-tac-toe_',
            'paper',
            'rsparse_dataset',
            'iris_multi',
            'tic-tac-toe__',
            'small',
            'small_']

INCLUDED = [
    'avila',
    'bank',
    'bean',
    'bidding',
    'eeg',
    'fault',
    'htru',
    'magic',
    'occupancy',
    'page',
    'raisin',
    'rice',
    'room',
    'segment',
    'skin',
    'wilt',]

In [ ]:
def extract_timed_results_from_base_dir(base_dir, contree=False):
    results_dir = []

    # Filter directories based on contree flag
    if contree:
        # Keep only directories starting with "contree"
        approaches = [name for name in os.listdir(base_dir)
                      if os.path.isdir(os.path.join(base_dir, name)) and name.startswith("contree")]
    else:
        # Keep all directories NOT starting with "contree"
        approaches = [name for name in os.listdir(base_dir)
                      if os.path.isdir(os.path.join(base_dir, name)) and not name.startswith("contree")]

    for approach in approaches:
        results_dir += extract_timed_results_dir(os.path.join(base_dir, approach))

    return results_dir

results_dir = extract_timed_results_from_base_dir(BASE_DIR, contree=False)
results_dir

In [ ]:
def compile_all_dir(results_dir, overwrite=False):
    compiled = os.path.join(BASE_DIR, "compiled.csv")

    if os.path.isfile(compiled) and not overwrite:
        return pd.read_csv(compiled)

    dfs = list()
    count = 1
    total = len(results_dir)
    for directory in results_dir:
        dfs.append(compile(directory, overwrite=overwrite))
        print(f"Compiled results for \t {count} / {total}")
        count += 1

    df = pd.concat(dfs, ignore_index=True)
    df.to_csv(compiled, index=None)
    return df

def compile(directory,overwrite=False):
    compiled_file = os.path.join(directory, "compiled.csv")
    if os.path.isfile(compiled_file) and not overwrite:
        return pd.read_csv(compiled_file)
    timeout = directory.split("/")[-1].split("_")[-1]
    method = directory.split("/")[-2]
    results = list()
    datasets = os.listdir(directory)
    for dataset in datasets:
        if dataset in EXCLUDED or dataset not in INCLUDED:
            continue
        path = os.path.join(directory, dataset)
        if not os.path.isdir(path):
            continue
        files = os.listdir(path)
        for file in files:
            if not file.endswith("json"):
                continue
            infos = file.split('.')[0].split("_")

            depth = int(infos[0])

            file_path = os.path.join(path, file)

            with open(file_path, "r") as f:
                data = json.load(f)
                size = len(data['runtimes'])

                for i in range(size):
                    results.append({
                        "name": dataset,
                        "support": data["support"],
                        "depth": depth,
                        "method": method,
                        "timeout":int(timeout),
                        "metric": i,
                        "runtime": data["runtimes"][i],
                        "cache_size": data["cache_size"][i],
                        "cache_hit": data["cache_hits"][i],
                        "error": data["errors"][i],
                    })

    df = pd.DataFrame(results)
    df.sort_values(["name", "depth", "method", "timeout", "runtime"], inplace=True)
    df.to_csv(f"{directory}/compiled.csv", index=None)
    return df


In [ ]:
df = compile_all_dir(results_dir, overwrite=True)

In [ ]:
def parse_original_contree(directory, timeout, overwrite=False):
    """
    Parse solver results where each line contains:
        Runtime: <float> Error: <int>
    Produces a DataFrame similar to parse_result_blossom.
    """
    result_file = os.path.join(directory, "compiled.csv")
    if os.path.exists(result_file) and not overwrite:
        return pd.read_csv(result_file)

    results = []
    method = directory.split("/")[-2]
    for dataset in os.listdir(directory):

        if dataset in EXCLUDED or dataset not in INCLUDED:
            continue
        data_path = os.path.join(directory, dataset)
        if not os.path.isdir(data_path):
            continue

        for file in os.listdir(data_path):
            path = os.path.join(data_path, file)
            if not path.endswith("txt"):
                continue

            # optional: extract depth from filename if present
            depth = int(file.split(".")[0]) if file.split(".")[0].isdigit() else 0

            try:
                with open(path, 'r') as f:
                    content = f.read()

                # Find all lines with runtime and error
                search_lines = re.findall(r"Runtime:\s*([\d.eE+-]+)\s*Error:\s*(\d+)", content)

                for i, (runtime_str, error_str) in enumerate(search_lines):
                    results.append({
                        "name": dataset,
                        "support": 1,
                        "depth": depth,
                        "method": method,
                        "timeout": timeout,
                        "metric": i,
                        "runtime": float(runtime_str),
                        "error": float(error_str),
                    })

            except Exception as e:
                print(f"Error processing {path}: {e}")

    df = pd.DataFrame(results)
    if not df.empty:
        df.sort_values(["name", "depth", "method", "timeout", "runtime"], inplace=True)
        df.to_csv(result_file, index=False)

    return df



In [ ]:
contree_dirs = extract_timed_results_from_base_dir(BASE_DIR, contree=True)
contree_dfs = list()
print(contree_dirs)
for directory in contree_dirs:
    #print(directory)
    endtime = directory.split("/")[-1].split("_")[-1]
    contree_dfs.append(parse_original_contree(directory, endtime, overwrite=True))

compiled = pd.concat(contree_dfs, ignore_index=True)
compiled.sort_values(["name", "depth", "method", "timeout", "runtime"], inplace=True)
compiled.to_csv(f"{BASE_DIR}/compiled_contree.csv", index=None)

In [ ]:
def all_df(base_dir, overwrite=False, use_contree=True):
    all_file = os.path.join(base_dir, "all.csv")
    if os.path.exists(all_file) and not overwrite:
        return pd.read_csv(all_file)
    results_dir = extract_timed_results_from_base_dir(base_dir)
    print(results_dir)
    df = compile_all_dir(results_dir, overwrite)
    #topn_df = create_topk_compilation(base_dir, overwrite)
    to_concat = [df]
    greedy_file = os.path.join(BASE_DIR, "greedy.csv")
    if os.path.exists(greedy_file):
        print("Greedy file exists")
        greedy = pd.read_csv(greedy_file)
        greedy = greedy[(greedy["name"].isin(INCLUDED)) & (greedy["depth"] >= 3)]
        greedy["timeout"] = 300
        to_concat.append(greedy)
    blossom_file = os.path.join(BASE_DIR, "compiled_contree.csv")
    if os.path.exists(blossom_file) and use_contree:
        blossom = pd.read_csv(blossom_file)
        to_concat.append(blossom)
    df = pd.concat(to_concat, ignore_index=True)
    df.sort_values(["name", "depth", "method", "timeout", "runtime"], inplace=True)
    df.to_csv(all_file, index=None)
    return df

In [ ]:
df = all_df(BASE_DIR, overwrite=True, use_contree=True)

In [ ]:
df.method.unique()

In [ ]:
# TODO : Consider contree here
def to_exclude(base_df, depth, max_time):
    run_max = list()
    for dataset in base_df["name"].unique():
        x = df[(df.method == "restart") & (base_df.depth == depth) & (base_df.timeout == 15) & (base_df.name == dataset)]
        if len(x) == 0:
            continue
        run_max.append(x.loc[x["runtime"].idxmax()].to_dict())
    if len(run_max) == 0:
        return []
    runs = pd.DataFrame(run_max)
    names = list(runs[runs.runtime <= max_time]["name"])
    return names


In [ ]:
def find_best_runtime(filtered_df):
    min_error = filtered_df["error"].min()
    filtered_df = filtered_df[filtered_df.error == min_error]
    best_row = filtered_df.loc[filtered_df["runtime"].idxmin()]
    return best_row.to_dict()

In [ ]:
def compute_gap(error, min_error):
    if error == min_error:
        return 0
    return np.abs(error - min_error) / max(error, min_error)

In [ ]:
def add_greedy_approaches(all_df, sub_df, depth, greedy_methods, time_limit, excluded_datasets):
    g_df = all_df[(~df['name'].isin(excluded_datasets)) & (all_df["method"].isin(greedy_methods)) & (all_df.depth == depth)]
    g_df = g_df.assign(timeout=time_limit)
    return pd.concat([sub_df, g_df], ignore_index=True)


In [ ]:
%matplotlib agg
GREEDY_METHODS = ["c4.5", "top3", "top5"]
def primal_gap(all_df, depths, time_limit, skip_plot, output_plot, exclusion_time):

    integral = list()
    greedy_gaps = list()
    for depth in depths:
        excluded_datasets = to_exclude(all_df, depth, exclusion_time)
        sub_df  = df.loc[(~df['name'].isin(excluded_datasets)) & (df.depth == depth) & (df.timeout == time_limit)].copy()
        sub_df = add_greedy_approaches(all_df, sub_df, depth, GREEDY_METHODS, time_limit, excluded_datasets)
        datasets = sub_df["name"].unique()
        approaches = sub_df["method"].unique()
        for dataset in datasets:
            os.makedirs(f"{output_plot}/{dataset}/figures", exist_ok=True)
            dataset_df = sub_df.loc[sub_df.name == dataset].copy()
            best_error = all_df.loc[(all_df.name == dataset) & (all_df.depth == depth)]["error"].min()
            dataset_df.loc[:, "gap"] = dataset_df["error"].apply(lambda row: compute_gap(row, best_error))
            if not skip_plot:
                plt.figure(figsize=(10, 6))

            for approach in approaches:
                method_df = dataset_df[dataset_df.method == approach].copy()
                if len(method_df) == 0:
                    print(approach, depth,  dataset, len(method_df))
                    continue

                first_line = pd.DataFrame(method_df.loc[method_df["runtime"].idxmin()]).transpose()
                first_line = first_line.assign(runtime=0, gap=1)
                method_df = pd.concat([first_line, method_df], ignore_index=True)

                end_time = int(np.ceil(method_df["runtime"].max()))
                if end_time < time_limit:
                    last_line = pd.DataFrame(method_df.loc[method_df["runtime"].idxmax()]).transpose()
                    last_line = last_line.assign(runtime=time_limit)
                    method_df = pd.concat([method_df, last_line], ignore_index=True)
                    # print(method_df)


                method_df.sort_values(["runtime"], inplace=True)


                if len(method_df) == 0:
                    continue
                # if approach in GREEDY_METHODS:
                # if len(method_df) == 1:
                #     method_df = pd.DataFrame(method_df.loc[method_df["runtime"].idxmax()]).transpose()
                #
                # # first_line = pd.DataFrame(method_df.loc[method_df["runtime"].idxmin()]).transpose()
                # # first_line = first_line.assign(runtime = 0, gap=100)
                # end_time = int(np.ceil(method_df["runtime"].max()))
                # time_to_append = time_limit
                # if end_time >= time_limit:
                #      time_to_append = end_time + 1
                # to_append = method_df.assign(runtime=time_to_append)
                # method_df = pd.concat([method_df, to_append], ignore_index=True)

                if approach not in GREEDY_METHODS and not skip_plot:
                    plt.step(method_df["runtime"], method_df["gap"], where="post", label=f"{approach}")
                    # print(approach, method_df["gap"][method_df["gap"].map(type) == np.float64], method_df["runtime"].dtype)
                    plt.fill_between(method_df["runtime"], method_df["gap"].astype(float), 0,  step="post", alpha=0.3)


                method_df["delta"] = method_df["runtime"].diff()
                method_df["p_prev"] = method_df["gap"].shift()
                method_df["increment"] = method_df["p_prev"] * method_df["delta"]
                method_df["P"] = method_df["increment"].cumsum().fillna(0)
                integral.append(method_df.loc[method_df["runtime"].idxmax()].to_dict())
                if approach in GREEDY_METHODS:
                    l = method_df.loc[method_df["runtime"].idxmax()].to_dict()
                    l["error"] = best_error
                    greedy_gaps.append(l)


            if not skip_plot:
                plt.xlabel('Time (s)')
                plt.ylabel('Primal Gap (%)')
                plt.title(f'Primal Gap Over Time for {dataset} and depth {depth}')
                plt.legend()
                plt.savefig(f"{output_plot}/{dataset}/figures/{depth}.png", dpi=400, bbox_inches='tight')
                plt.clf()
                plt.close()


    primal_df = pd.DataFrame(integral)
    primal_df.drop(columns=["metric","cache_size", "cache_hit",  "delta", "p_prev", "increment"], inplace=True)
    primal_df["P_ratio"] = primal_df["P"] / primal_df["timeout"]
    primal_df.to_csv(f"{BASE_DIR}/primal_{time_limit}.csv", index=None)

    g = pd.DataFrame(greedy_gaps)
    #g.to_csv(f"{BASE_DIR}/greedy_gaps.csv", index=None)

    return primal_df, g



In [ ]:
RUNTIMES = [5, 15, 30, 60, 120, 300, 600]
def all_primal(df, skip_plot=True, overwrite=False):
    path = os.path.join(BASE_DIR, "all_primal.csv")
    if os.path.exists(path) and not overwrite:
        return pd.read_csv(path)
    gaps = list()
    greees = list()
    depths = df["depth"].unique()
    print(f"All depths : {depths}")
    for runtime in RUNTIMES:
        print(runtime)
        pdf, gree = primal_gap(df, depths, runtime, skip_plot, f"{BASE_DIR}/results_{runtime}", 1)
        greees.append(gree)
        gaps.append(pdf)
    primal_df = pd.concat(gaps, ignore_index=True)
    gdf = pd.concat(greees, ignore_index=True)
    gdf.to_csv(f"{BASE_DIR}/greee.csv")
    primal_df.to_csv(path, index=None)
    return primal_df


In [ ]:
primal_df = all_primal(df, skip_plot=True, overwrite=True)

In [ ]:
def average_primal(primal_df, ratio=True):
    primals = list()
    for timeout in RUNTIMES:

        pdf = primal_df.loc[primal_df.timeout == timeout].copy()
        pdf = pdf[pdf["name"].isin(INCLUDED)]
        pdf.drop(columns=["name", "runtime"], inplace=True)
        x = pdf.groupby(["method", "depth"]).mean().reset_index().sort_values(["P"])
        primals.append(x.copy())

    pdf = pd.concat(primals, ignore_index=True)
    pdf["timeout"] = pdf["timeout"].astype(int)
    pivoted = pdf.pivot_table(
        index=['method', 'depth'],
        columns='timeout',
        values='P_ratio' if ratio else  'P'
    ).reset_index()
    pivoted.sort_values(["depth"] + RUNTIMES, inplace=True)
    name = f"{BASE_DIR}/avg_primal_p_ratio_pivoted.csv" if ratio else f"{BASE_DIR}/avg_primal_p_pivoted.csv"
    pivoted.to_csv(name, index=None)
    return pivoted

In [ ]:
average_primal(primal_df, ratio=False)
average_primal(primal_df, ratio=True)

In [ ]:
ratio = True
file = f"{BASE_DIR}/avg_primal_p_ratio_pivoted.csv" if ratio else f"{BASE_DIR}/avg_primal_p_pivoted.csv"
compare_df = pd.read_csv(file)
methods = ["first_heuristic_yes", "contree_heuristic_yes", "contree_heuristic_no" , "c4.5"]
#methods = compare_df["method"].unique()
plt.figure(figsize=(6, 5))
fig, ax = plt.subplots()
for method in methods:
    m_df = compare_df[(compare_df.method == method) & (compare_df.depth < 9 )]
    ax.plot(m_df["depth"].astype(int), m_df["15"] * 100, label=f"{method}")
plt.xlabel("Depth", fontsize=14)
ax.set_xticks(m_df["depth"])
plt.ylabel("Average primal integral fraction (%)", fontsize=14)
plt.grid(True)
plt.legend(["LDS-Contree", "Contree-Gini", "Contree"])
#plt.legend()
plt.savefig(f"{BASE_DIR}/figure_compare_avg_integral.pdf", dpi=400, bbox_inches='tight')
plt.show()



In [ ]:
compare_df["method"].unique()

In [ ]:
list(methods)

In [ ]:
depth = 5
ratio = True
methods = ["first_heuristic_yes", "contree_heuristic_yes", "contree_heuristic_no", "c4.5"]

from collections import Counter
labels = {
    "first_heuristic_yes": "Contree-Lds",
    "contree_heuristic_yes": "Contree-Gini",
    "contree_heuristic_no": "Contree",
    "c4.5": "C4.5"
}
def rename(row):
    if row in labels.keys():
        return labels[row]
    return row.capitalize()

def split_method(method):
    if '-' in method:
        return pd.Series(method.split('-'))
    else:
        return pd.Series([method, ''])

file = f"{BASE_DIR}/avg_primal_p_ratio_pivoted.csv" if ratio else f"{BASE_DIR}/avg_primal_p_pivoted.csv"
pivoted = pd.read_csv(file)
pivoted = pivoted[pivoted["method"].isin(methods)]
RUNTIMES_STR = [str(i) for i in RUNTIMES]
pivot = pivoted[(pivoted.depth == depth) ]
# & ~pivoted.method.isin(['gainlds-luby', 'gainlds-exponential', 'gainlds-monotonic'])
pivot[['approach', 'subapproach']] = pivot['method'].apply(split_method)
pivot['approach'] = pd.Categorical(pivot['approach'], categories=methods, ordered=True)
pivot["approach"] = pivot['approach'].apply(rename)

# Bold the minimum per column
for col in RUNTIMES_STR:
    min_val = pivot[col].min()
    if ratio:
        min_val *= 100
        pivot[col] = pivot[col] * 100
    pivot[col] = pivot[col].apply(lambda x: f"\\textbf{{{x:.1f}}}" if np.isclose(x, min_val) else f"{x:.1f}")

# Order columns and sort
pivot = pivot[['approach', 'subapproach'] + RUNTIMES_STR]
pivot = pivot.sort_values(by=['approach', 'subapproach'])

# Count how many rows per approach (for multirow)
approach_counts = Counter(pivot['approach'])

# Create LaTeX lines
latex_lines = []
num_runtimes = len(RUNTIMES_STR)

latex_lines.append("\\begin{tabular}{ll" + "r" * num_runtimes + "}")
latex_lines.append("\\toprule")
latex_lines.append(" & & \\multicolumn{" + str(num_runtimes) + "}{c}{Runtime (s)} \\\\")
latex_lines.append("Approach & Sub & " + " & ".join(str(i) for i in RUNTIMES_STR) + " \\\\")
latex_lines.append("\\midrule")

last_approach = None
for i, (_, row) in enumerate(pivot.iterrows()):
    approach = row['approach']
    sub = row['subapproach'] or '--'
    times = [row[col] for col in RUNTIMES_STR]

    # If new approach, insert \midrule separator (except first group)
    if approach != last_approach and last_approach is not None:
        latex_lines.append("\\midrule")

    if approach != last_approach:
        rowspan = approach_counts[approach]
        latex_lines.append(f"\\multirow{{{rowspan}}}{{*}}{{{approach}}} & {sub} & " + " & ".join(times) + " \\\\")
        last_approach = approach
    else:
        latex_lines.append(f"& {sub} & " + " & ".join(times) + " \\\\")

latex_lines.append("\\bottomrule")
latex_lines.append("\\end{tabular}")

print("\n".join(latex_lines))
